# Tutorial 4 — Ranking: a second opinion on the shortlist

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_04_ranking.ipynb)

Companion to the *Ranking* section. Here you will:

1. take ALS's top-100 candidates for a set of users and build the seven features the post lists,
2. train the LambdaMART reranker **twice**: once with the positive-injection bug from the first version of the project, once fixed,
3. probe both models to see what each actually learned (the buggy one promotes the lowest-scored, least popular candidate),
4. evaluate both on the *identical* candidate sets for held-out users, and inspect single users — including the "position 98 → position 1" kind of case.

CPU is fine for this one (~3 min).

In [ ]:
#@title Setup — clone the repo, install deps, download the pre-computed artifacts (~1 min)
import os, sys, urllib.request
REPO_URL = "https://github.com/juanmigutierrez/generative-recommendation-engine"
RELEASE  = REPO_URL + "/releases/download/v1.0-artifacts"
QUICK    = os.environ.get("TUTORIAL_QUICK") == "1"   # tiny sizes for headless smoke tests

if not os.path.exists("backend"):
    if not os.path.exists("generative-recommendation-engine"):
        !git clone -q {REPO_URL}
    %cd generative-recommendation-engine
if "google.colab" in sys.modules:
    !pip install -q implicit lightgbm pyarrow ipywidgets 2>&1 | tail -1
os.makedirs("data/processed", exist_ok=True)

def fetch(*names):
    """Download an artifact from the GitHub release unless it is already on disk."""
    for n in names:
        p = os.path.join("data", "processed", n)
        if not os.path.exists(p):
            print("downloading", n, "...")
            urllib.request.urlretrieve(f"{RELEASE}/{n}", p)

fetch("train.parquet", "item_catalog.parquet", "train_sequences.parquet", "val_targets.parquet", "item_embeddings.npy")
for p in ["backend", "backend/scripts"]:
    if p not in sys.path: sys.path.insert(0, p)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
pd.set_option("display.max_colwidth", 90)
P = os.path.join("data", "processed")
print("ready")

## 1. Retrieval gives 100 candidates per user

ALS is the retrieval stage here (the same reasoning as in the project: it scores the whole catalog in one matrix multiply).

In [ ]:
import time, lightgbm as lgb
from models.als import ALSRecommender
from models.ranking_features import FEATURE_COLUMNS, build_item_features, build_user_profile_embeddings, build_user_interaction_counts, assemble_features
from models.metrics import recall_at_k, ndcg_at_k

train = pd.read_parquet(f"{P}/train.parquet"); items = pd.read_parquet(f"{P}/item_catalog.parquet").set_index("item_id"); title = items["description"]
train_seq = pd.read_parquet(f"{P}/train_sequences.parquet"); emb = np.load(f"{P}/item_embeddings.npy")
val_targets = {r.user_id: set(r.item_ids) for r in pd.read_parquet(f"{P}/val_targets.parquet").itertuples()}
n_items = len(items); n_users = int(max(train.user_id.max(), max(val_targets)) + 1)
t0 = time.time(); als = ALSRecommender(factors=64, regularization=0.05, iterations=15, alpha=40.0).fit(train, n_users, n_items); print(f"ALS fit {time.time()-t0:.0f}s")

item_features = build_item_features(train, n_items); user_profiles = build_user_profile_embeddings(train_seq, emb); user_n = build_user_interaction_counts(train_seq)
eligible = sorted(set(train_seq.user_id) & set(val_targets)); rng = np.random.RandomState(42)
N_TRAIN, N_EVAL = (1500, 400) if QUICK else (12000, 3000)
chosen = rng.choice(eligible, N_TRAIN + N_EVAL, replace=False); train_users, eval_users = np.sort(chosen[:N_TRAIN]), np.sort(chosen[N_TRAIN:])

def candidates(users, N=100):
    ids, scores = als.model.recommend(users, als.user_items[users], N=N, filter_already_liked_items=True)
    return [list(r) for r in ids], [list(r) for r in scores]
print(f"{N_TRAIN:,} ranker-training users, {N_EVAL:,} held-out users; 100 ALS candidates each")

## 2. Build the feature table — with and without the bug

Seven features per (user, candidate): ALS score, log popularity, price (+ has_price flag), recency, content similarity to the user's profile embedding, log history length. The label is 1 if the candidate is one of the user's val items.

**The bug:** ALS's top-100 contains the true item for only ~13% of users. The first version "fixed" that by *injecting* the missing true items with a placeholder ALS score (the list minimum). Toggle it below.

In [ ]:
def build_table(users, inject):
    ids, scores = candidates(users); n_inj = 0
    for k, u in enumerate(users):
        missing = val_targets[u] - set(ids[k])
        if inject and missing:
            n_inj += len(missing); ids[k] += list(missing); scores[k] += [min(scores[k])] * len(missing)
    df = assemble_features(users, ids, scores, item_features, user_profiles, user_n, emb)
    pos = {(u, i) for u in users for i in val_targets[u]}
    df["label"] = [int((u, i) in pos) for u, i in zip(df.user_id, df.item_id)]
    if not inject:   # groups with no positive carry no LambdaRank gradient — drop them
        keep = df.groupby("user_id")["label"].transform("max") == 1; df = df[keep]
    return df.sort_values("user_id").reset_index(drop=True), n_inj

def fit_ranker(df):
    r = lgb.LGBMRanker(objective="lambdarank", metric="ndcg", n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=20, random_state=42, verbosity=-1)
    r.fit(df[FEATURE_COLUMNS].values, df["label"].values, group=df.groupby("user_id").size().values); return r

df_bug, n_inj = build_table(train_users, inject=True)
print(f"WITH injection: {len(df_bug):,} rows, {df_bug.label.sum():,} positives of which {n_inj:,} injected ({n_inj/df_bug.label.sum():.0%})")
df_fix, _ = build_table(train_users, inject=False)
print(f"FIXED:          {len(df_fix):,} rows, {df_fix.label.sum():,} positives, {df_fix.user_id.nunique():,} users kept (the ones whose true item ALS actually retrieved)")
t0 = time.time(); ranker_bug = fit_ranker(df_bug); ranker_fix = fit_ranker(df_fix); print(f"both rankers trained in {time.time()-t0:.0f}s")

## 3. What did each one learn?

Hold every feature at a typical value and sweep one. The buggy ranker's response to the ALS score and to popularity is *inverted*: it learned that the injected item — lowest ALS score, zero training popularity — is the positive.

In [ ]:
base = dict(als_score=0.5, item_popularity_log=np.log1p(50), item_price_imputed=30.0, has_price=1.0, item_recency_norm=0.8, content_sim=0.7, user_n_interactions_log=np.log1p(4))
def sweep(model, feat, values):
    rows = []
    for v in values:
        d = dict(base); d[feat] = v; rows.append(float(model.booster_.predict(np.array([[d[c] for c in FEATURE_COLUMNS]]))[0]))
    return rows

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
for ax, (feat, xs, lab) in zip(axes, [("als_score", np.linspace(0, 1.3, 14), "ALS score"), ("item_popularity_log", np.log1p([0, 1, 5, 20, 100, 500, 2000]), "log(1 + train count)"), ("content_sim", np.linspace(0, 1, 11), "content similarity")]):
    ax.plot(xs, sweep(ranker_bug, feat, xs), color="#eb6834", lw=2, label="with injection (bug)"); ax.plot(xs, sweep(ranker_fix, feat, xs), color="#2a78d6", lw=2, label="fixed")
    ax.set_xlabel(lab); ax.set_ylabel("ranker score"); [ax.spines[s].set_visible(False) for s in ["top", "right"]]
axes[0].legend(frameon=False); plt.tight_layout(); plt.show()

## 4. Same candidates, three orders

For the held-out users: ALS's own order, the buggy reranker, the fixed reranker — all on the identical top-100.

In [ ]:
ids_e, scores_e = candidates(eval_users)
df_e = assemble_features(eval_users, ids_e, scores_e, item_features, user_profiles, user_n, emb)
df_e["bug"] = ranker_bug.booster_.predict(df_e[FEATURE_COLUMNS].values); df_e["fix"] = ranker_fix.booster_.predict(df_e[FEATURE_COLUMNS].values)
orders = {"ALS order": {u: ids_e[k] for k, u in enumerate(eval_users)}}
for col in ["bug", "fix"]:
    orders[{"bug": "reranked (bug)", "fix": "reranked (fixed)"}[col]] = {u: g.sort_values(col, ascending=False).item_id.tolist() for u, g in df_e.groupby("user_id")}
res = {name: {f"recall@{k}": np.mean([recall_at_k(lst[u], val_targets[u], k) for u in eval_users]) for k in (10, 20)} | {f"ndcg@{k}": np.mean([ndcg_at_k(lst[u], val_targets[u], k) for u in eval_users]) for k in (10, 20)} for name, lst in orders.items()}
display(pd.DataFrame(res).T.round(4))

### Inspect one user

Users where the fixed reranker moved a true item into the top 10 from deep in ALS's list. The feature values show *why* — usually content similarity.

In [ ]:
moved = []
for u in eval_users:
    a, f = orders["ALS order"][u], orders["reranked (fixed)"][u]
    for t in val_targets[u]:
        if t in a and a.index(t) >= 20 and f.index(t) < 10: moved.append((u, t, a.index(t) + 1, f.index(t) + 1))
print(f"{len(moved)} (user, item) cases moved from ≥ position 20 into the top 10")

def inspect(case):
    u, t, pa, pf = case
    print("history:", " | ".join(title[i][:30] for i in train_seq.set_index('user_id').loc[u, 'item_ids'][-5:]))
    print(f"\ntrue next item: {title[t][:80]}\n  ALS position {pa} → reranked position {pf}")
    row = df_e[(df_e.user_id == u) & (df_e.item_id == t)].iloc[0]
    print("  features:", {c: round(float(row[c]), 3) for c in FEATURE_COLUMNS})
    print("\nreranked top-5:"); [print(f"  {'✔' if i in val_targets[u] else ' '} {r}. {title[i][:70]}") for r, i in enumerate(orders['reranked (fixed)'][u][:5], 1)]

if moved: interact(inspect, case=widgets.Dropdown(options=[(f"user {u}: {pa} → {pf}", c) for c in moved[:40] for (u, t, pa, pf) in [c]], description="case"))

**Next:** [Tutorial 5 — Evaluation](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_05_evaluation.ipynb): every model on the papers' protocol, with SASRec.